In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the training dataset
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/media_campaign_cost/train.csv'
train_df = pd.read_csv(train_path)

# Display the first few rows of the dataset
print(train_df.head())

# Display the summary of the dataset
print(train_df.info())

# Display the statistical summary of the numerical columns
print(train_df.describe())

# Check for missing values
print(train_df.isnull().sum())

# Distinguish column types
numeric_features = train_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = train_df.select_dtypes(exclude=[np.number]).columns.tolist()

# Visualize the distribution of numerical features
for feature in numeric_features:
    plt.figure(figsize=(10, 6))
    sns.histplot(train_df[feature], kde=True)
    plt.title(f'Distribution of {feature}')
    plt.show()

# Visualize the distribution of categorical features
for feature in categorical_features:
    plt.figure(figsize=(10, 6))
    sns.countplot(data=train_df, x=feature)
    plt.title(f'Distribution of {feature}')
    plt.xticks(rotation=45)
    plt.show()

# Check the correlation between numerical features
correlation_matrix = train_df[numeric_features].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Numerical Features')
plt.show()


       id  store_sales  unit_sales  ...  prepared_food  florist    cost
0  177400         8.49           3  ...              0        0  133.42
1   35064         4.47           3  ...              0        0  109.06
2  268168         6.84           3  ...              1        1   91.58
3  177181         4.54           2  ...              0        0  124.36
4  271461         8.34           3  ...              1        1   96.55

[5 rows x 17 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28827 entries, 0 to 28826
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    28827 non-null  int64  
 1   store_sales           28827 non-null  float64
 2   unit_sales            28827 non-null  int64  
 3   total_children        28827 non-null  int64  
 4   num_children_at_home  28827 non-null  int64  
 5   avg_cars_at_home      28827 non-null  int64  
 6   gross_weight          2

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df)
print("column_info")
print(column_info)


2025-08-31 14:26:22.791 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': [], 'Numeric': ['id', 'store_sales', 'unit_sales', 'total_children', 'num_children_at_home', 'avg_cars_at_home', 'gross_weight', 'recyclable_package', 'low_fat', 'units_per_case', 'store_sqft', 'coffee_bar', 'video_store', 'salad_bar', 'prepared_food', 'florist', 'cost'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, StandardScale
import pandas as pd

# Load the test dataset
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/media_campaign_cost/test.csv'
test_df = pd.read_csv(test_path)

# Copy the datasets to avoid modifying the original data
train_df_copy = train_df.copy()
test_df_copy = test_df.copy()

# Handle missing values
fill_missing = FillMissingValue(features=train_df_copy.columns, strategy='mean')
train_df_processed = fill_missing.fit_transform(train_df_copy)
test_df_processed = fill_missing.transform(test_df_copy)

# Encode categorical variables (if any)
# Since there are no categorical variables, this step is skipped

# Scale features
scale_features = StandardScale(features=train_df_processed.columns.difference(['cost']))
train_df_processed = scale_features.fit_transform(train_df_processed)
test_df_processed = scale_features.transform(test_df_processed)

# Display the processed datasets
train_df_processed.head(), test_df_processed.head()


(         id  store_sales  unit_sales  ...  prepared_food   florist    cost
 0 -0.018151     0.649551   -0.052892  ...      -1.003579 -1.006788  133.42
 1 -1.392321    -0.568748   -0.052892  ...      -1.003579 -1.006788  109.06
 2  0.858160     0.149503   -0.052892  ...       0.996433  0.993258   91.58
 3 -0.020265    -0.547533   -1.325621  ...      -1.003579 -1.006788  124.36
 4  0.889952     0.604092   -0.052892  ...       0.996433  0.993258   96.55
 
 [5 rows x 17 columns],
          id  store_sales  unit_sales  ...  prepared_food   florist    cost
 0 -0.054761    -1.165775   -1.325621  ...      -1.003579  0.993258   75.44
 1  0.993766    -0.741491   -0.052892  ...       0.996433  0.993258  131.75
 2 -1.670995    -0.832409   -0.052892  ...      -1.003579 -1.006788   59.19
 3  0.641486     2.489122    1.219837  ...      -1.003579 -1.006788   86.42
 4 -1.386480    -1.396100   -0.052892  ...      -1.003579 -1.006788  123.63
 
 [5 rows x 17 columns])

In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Using the processed training data DataFrame
column_info = get_column_info(train_df_processed)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['id', 'store_sales', 'unit_sales', 'total_children', 'num_children_at_home', 'avg_cars_at_home', 'gross_weight', 'recyclable_package', 'low_fat', 'units_per_case', 'store_sqft', 'coffee_bar', 'video_store', 'salad_bar', 'prepared_food', 'florist', 'cost'], 'Datetime': [], 'Others': []}


In [5]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from metagpt.tools.libs.data_preprocess import OneHotEncode

# Since there are no categorical features, we don't need to perform OneHotEncoding.
# If there were, we would do it like this:
# one_hot = OneHotEncode(features=categorical_features)
# train_df_processed = one_hot.fit_transform(train_df_processed)
# test_df_processed = one_hot.transform(test_df_processed)

# Split the data into features and target
X_train = train_df_processed.drop(columns=['cost'])
y_train = train_df_processed['cost']
X_test = test_df_processed.drop(columns=['cost'])

# Split the training data into training and validation sets
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Initialize the XGBoost model
xgb_model = XGBRegressor(objective='reg:squarederror', random_state=42)

# Define the parameter grid for GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}

# Perform grid search to find the best hyperparameters
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train_split, y_train_split)

# Get the best model
best_model = grid_search.best_estimator_

# Evaluate the best model on the validation set
y_val_pred = best_model.predict(X_val_split)
val_rmse = np.sqrt(mean_squared_error(y_val_split, y_val_pred))
print(f'Validation RMSE: {val_rmse}')

# Predict on the test set
y_test_pred = best_model.predict(X_test)

# Save the predictions
test_df['cost'] = y_test_pred
test_df[['id', 'cost']].to_csv('predicted_media_campaign_cost.csv', index=False)

# Calculate and report the rmlse on the test set
# Assuming the test set has the true 'cost' values
if 'cost' in test_df.columns:
    true_cost = test_df['cost']
    predicted_cost = y_test_pred
    rmlse = np.sqrt(mean_squared_error(np.log(true_cost + 1), np.log(predicted_cost + 1)))
    print(f'Test RMLSE: {rmlse}')
else:
    print('Test set does not contain true cost values, cannot calculate RMLSE.')


Validation RMSE: 29.179029483836757
Test RMLSE: 0.0
